In [ ]:
#!pip install -r requirements.txt
!python app.py

^C


: 

In [5]:
# Run the Action Recognition API
# Note: This will start the server and block the cell until you stop it (Ctrl+C)

import subprocess
import sys
import os

# Check if we're in the right directory
print("📍 Current directory:", os.getcwd())

# Check if model files exist
required_files = [
    'trained_action_recognition_model.keras',
    'model_metadata.json', 
    'app.py'
]

print("\n🔍 Checking required files...")
all_files_exist = True
for file in required_files:
    if os.path.exists(file):
        print(f"✅ {file}")
    else:
        print(f"❌ {file} - MISSING!")
        all_files_exist = False

if not all_files_exist:
    print("\n💡 Please make sure you have:")
    print("1. Trained the model in the LIME notebook")
    print("2. Saved the model using the model saving cells")
    print("3. All files are in the same directory")
else:
    print("\n🚀 Starting API server...")
    print("📝 Documentation: http://localhost:8000/docs")
    print("🌐 Test client: Open test_client.html in browser")
    print("⏹️  Use Kernel -> Interrupt to stop the server")
    print("-" * 50)
    
    # Start the API server
    try:
        # Use the simple launcher to avoid reload issues
        subprocess.run([sys.executable, "run_api.py"])
    except KeyboardInterrupt:
        print("\n👋 API server stopped")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        print("\n🔧 Alternative: Run this in terminal instead:")
        print("   cd M:/LIME")
        print("   python run_api.py")

📍 Current directory: m:\LIME

🔍 Checking required files...
✅ trained_action_recognition_model.keras
✅ model_metadata.json
✅ app.py

🚀 Starting API server...
📝 Documentation: http://localhost:8000/docs
🌐 Test client: Open test_client.html in browser
⏹️  Use Kernel -> Interrupt to stop the server
--------------------------------------------------


In [7]:
# Alternative: Start API in background (non-blocking)
# This allows you to continue using the notebook while API runs

import subprocess
import time
import requests
import os
from threading import Thread

def start_api_background():
    """Start the API in background"""
    try:
        # Start the API process
        process = subprocess.Popen(
            [os.sys.executable, "run_api.py"],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            cwd=os.getcwd()
        )
        
        print("🚀 API starting in background...")
        time.sleep(3)  # Give it time to start
        
        # Test if API is running
        try:
            response = requests.get("http://localhost:8000/health", timeout=5)
            if response.status_code == 200:
                print("✅ API is running successfully!")
                print("📝 Documentation: http://localhost:8000/docs")
                print("🌐 Test client: Open test_client.html")
                print(f"🔧 Process ID: {process.pid}")
                print("\n💡 To stop the API, run the next cell or restart the kernel")
                return process
            else:
                print("❌ API started but not responding correctly")
                process.terminate()
                return None
        except requests.exceptions.RequestException:
            print("❌ API failed to start or not accessible")
            process.terminate()
            return None
            
    except Exception as e:
        print(f"❌ Error starting API: {e}")
        return None

# Check files first
if not all(os.path.exists(f) for f in ['app.py', 'trained_action_recognition_model.keras', 'model_metadata.json']):
    print("❌ Required files missing. Please run the model training cells first.")
else:
    # Store the process globally so we can stop it later
    api_process = start_api_background()

🚀 API starting in background...
❌ API failed to start or not accessible


In [1]:
# Stop the background API process
import subprocess
import signal
import os

def stop_api():
    """Stop the background API process"""
    try:
        if 'api_process' in globals() and api_process is not None:
            print("🛑 Stopping API process...")
            api_process.terminate()
            api_process.wait()
            print("✅ API stopped successfully")
        else:
            print("ℹ️  No background API process found")
            print("🔍 Trying to stop any running Python processes on port 8000...")
            
            # Alternative: Kill any process using port 8000
            if os.name == 'nt':  # Windows
                subprocess.run(['netstat', '-ano', '|', 'findstr', ':8000'], shell=True)
                print("💡 Use Task Manager to manually stop Python processes if needed")
            else:  # Unix/Linux/Mac
                subprocess.run(['lsof', '-ti:8000', '|', 'xargs', 'kill', '-9'], shell=True)
                
    except Exception as e:
        print(f"❌ Error stopping API: {e}")
        print("💡 You may need to restart the kernel to fully stop the API")

# Test if API is still running
try:
    import requests
    response = requests.get("http://localhost:8000/health", timeout=2)
    print("🟡 API is currently running")
    print("Click 'Run' to stop it")
except:
    print("✅ No API detected on port 8000")

# Uncomment the next line to stop the API
stop_api()

🟡 API is currently running
Click 'Run' to stop it
ℹ️  No background API process found
🔍 Trying to stop any running Python processes on port 8000...
💡 Use Task Manager to manually stop Python processes if needed
💡 Use Task Manager to manually stop Python processes if needed


In [ ]:
# Test the API with a sample prediction
import requests
import json
from PIL import Image
import io
import os

def test_api():
    """Test the API with a sample image"""
    
    # Check if API is running
    try:
        response = requests.get("http://localhost:8000/health", timeout=3)
        if response.status_code != 200:
            print("❌ API is not responding. Please start it first.")
            return
    except:
        print("❌ API is not accessible. Please start it first.")
        print("💡 Run the background API cell above first")
        return
    
    print("✅ API is running! Testing with sample prediction...")
    
    # Get model info
    try:
        response = requests.get("http://localhost:8000/model-info")
        model_info = response.json()
        print(f"\n🤖 Model Info:")
        print(f"   Accuracy: {model_info['model_accuracy']}")
        print(f"   Classes: {len(model_info['action_classes'])}")
        print(f"   Actions: {', '.join(model_info['action_classes'][:5])}...")
    except Exception as e:
        print(f"❌ Error getting model info: {e}")
        return
    
    # Create a simple test image if no real image is available
    print(f"\n📸 Looking for test images...")
    
    # Check for sample images in VOC dataset
    sample_paths = [
        'M:/LIME/VOCdevkit/VOC2012/JPEGImages/2007_000027.jpg',
        'M:/LIME/VOCdevkit/VOC2012/JPEGImages/2007_000032.jpg',
        'M:/LIME/VOCdevkit/VOC2012/JPEGImages/2007_000033.jpg'
    ]
    
    test_image_path = None
    for path in sample_paths:
        if os.path.exists(path):
            test_image_path = path
            break
    
    if test_image_path:
        print(f"📷 Using sample image: {os.path.basename(test_image_path)}")
        
        # Test prediction
        try:
            with open(test_image_path, 'rb') as f:
                files = {'file': f}
                response = requests.post("http://localhost:8000/predict", files=files)
            
            if response.status_code == 200:
                result = response.json()
                print(f"\n🎯 Prediction Results:")
                print(f"   Predicted Action: {result['predicted_action']}")
                print(f"   Confidence: {result['confidence']:.3f}")
                print(f"   Top 3 predictions:")
                for i, pred in enumerate(result['top_predictions'][:3]):
                    print(f"     {i+1}. {pred['action']}: {pred['confidence']:.3f}")
                
                print(f"\n✅ API is working correctly!")
                print(f"🌐 Try the web interface: Open test_client.html")
                print(f"📝 Full documentation: http://localhost:8000/docs")
                
            else:
                print(f"❌ Prediction failed: {response.status_code}")
                print(f"   Error: {response.text}")
                
        except Exception as e:
            print(f"❌ Error testing prediction: {e}")
    else:
        print("⚠️  No test images found in VOC dataset")
        print("💡 You can still test manually:")
        print("   1. Open test_client.html in browser")
        print("   2. Upload any image to test the API")
        print("✅ API is ready for testing!")

# Run the test
test_api()